### Imports
> Before you begin coding, change the Runtime to a TPU!

In [ ]:
!pip install -qqq jaxtyping beartype wandb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.6/25.6 MB 102.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.6/208.6 kB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 451.6/451.6 kB 43.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.8/62.8 kB 7.3 MB/s eta 0:00:00


In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import flax.nnx as nnx
import optax
import orbax
import orbax.checkpoint as ocp
from datasets import load_dataset
from dataclasses import dataclass
from jaxtyping import Float, Integer, jaxtyped, Array
from beartype import beartype
from transformers import GPT2TokenizerFast
from einops import rearrange
from tqdm.autonotebook import tqdm

/usr/local/lib/python3.12/dist-packages/jax/_src/cloud_tpu_init.py:86: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### Downloading our Dataset and Tokenizer

View examples here: https://huggingface.co/datasets/Skylion007/openwebtext/viewer

In [ ]:
ds = load_dataset("Skylion007/openwebtext", split="train")

README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/80 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/80 [00:00<?, ?it/s]

plain_text/train-00000-of-00080.parquet:   0%|          | 0.00/303M [00:00<?, ?B/s]

plain_text/train-00001-of-00080.parquet:   0%|          | 0.00/306M [00:00<?, ?B/s]

...

plain_text/train-00077-of-00080.parquet:   0%|          | 0.00/303M [00:00<?, ?B/s]

plain_text/train-00078-of-00080.parquet:   0%|          | 0.00/301M [00:00<?, ?B/s]

plain_text/train-00079-of-00080.parquet:   0%|          | 0.00/303M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/8013769 [00:00<?, ? examples/s]

Loading dataset shards:   0%|          | 0/80 [00:00<?, ?it/s]

In [ ]:
tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
sentence = input("Write a random sentence: ")
tokens = tokenizer(sentence, return_tensors="np")
print(f"Token IDs: {tokens['input_ids'][0]}")

Write a random sentence: Pancakes and bananas
Token IDs: [   47  1192  1124   290 35484]


### GPT2-124M Configuration

In [ ]:
@dataclass(frozen=True, kw_only=True, slots=True)
@jaxtyped(typechecker=beartype)
class GPT2Config:
    max_seq_len: int = 1024
    vocab_size: int = 50257
    n_layers: int = 12
    n_head: int = 12
    hidden_size: int = 768
    mlp_hidden_size: int = 4 * 768

## Breaking Attention Into Blocks

<div align="center">
  <img src="https://machinelearningmastery.com/wp-content/uploads/2021/08/attention_research_1.png"
       alt="Image" width="300" height="400" />
</div>

The attention mechanism lies within the transformer architecture, meaning the picture below is the general architecture to how we will build our GPT-2 model. At first, the image looks like a lot - so we'll break down each part step-by-step.

* **Positional Embeddings:** Values appended to the embeddings, which are important because the attention architecture doesn't involve position to begin with. You can see how these embeddings looked, plotted, [here](https://en.wikipedia.org/wiki/Softmax_function).
* **Multi-Head Attention:** Where the attention mechanism takes place, using the KQV formula below:

$$
Attention(K, Q, V) = softmax(\frac{QK^T}{\sqrt{d}})V
$$

* Aside from simply applying this equation, our model will split the input embeddings into seperate "attention" heads. The splitting allows for attention in each head to focus on different features of our input.

<div align="center">
    <img src="https://jalammar.github.io/images/t/transformer_multi-headed_self-attention-recap.png" alt="Image" width="450" height="300" />
</div>

* **Add & Norm:** After the attention process, we always have a "Add & Norm" block. In code it looks fairly simple, involving [layer normalization](https://arxiv.org/abs/1607.06450) and a [residual connection](https://en.wikipedia.org/wiki/Residual_neural_network): `Output = LayerNorm(X + Attention(X))`
* **Feed Foward:** Throughout the transformer, these blocks are MLPs which are simply a sequence of dense layers. Most of the weights (and knowledge learned) in the model are located here.
* **Softmax & Linear:** Finally, at the end of our model, we apply a single linear layer and [softmax](https://en.wikipedia.org/wiki/Softmax_function) function to create a probability distribution over what word to predict next.

<div align="center">
  <img src="https://jalammar.github.io/images/t/transformer_decoder_output_softmax.png"
       alt="Image" width="400" height="300" />
</div>

### Multi-Head Self-Attention

As previously mentioned, the multi-head self attention applies the attention mechanism while splitting into seperate heads. The process is as simple as apportioning different parts of our input to different attention heads in set intervals, defined by `batch_size / n_head`.

<div align="center">
    <img src="https://jalammar.github.io/images/t/transformer_attention_heads_z.png" alt="Image" width="600" height="300" />
</div>

<div align="center">
    <img src="https://raw.githubusercontent.com/chloeli-15/ARENA_img/main/img/transformer-attn-new-v2.png" alt="Image" width="600" height="300" />
</div>

In [ ]:
class MultiHeadSelfAttention(nnx.Module):
    def __init__(self, config: GPT2Config, rngs: nnx.Rngs):
        self.n_heads = config.n_head
        self.head_size = config.hidden_size // config.n_head
        self.output_size = config.hidden_size
        self.hidden_size = config.hidden_size

        assert self.hidden_size % self.n_heads == 0

        self.Wq = nnx.Linear(self.hidden_size, self.hidden_size, use_bias=False, rngs=rngs)
        self.Wk = nnx.Linear(self.hidden_size, self.hidden_size, use_bias=False, rngs=rngs)
        self.Wv = nnx.Linear(self.hidden_size, self.hidden_size, use_bias=False, rngs=rngs)
        self.Wo = nnx.Linear(self.hidden_size, self.hidden_size, use_bias=False, rngs=rngs)

    @jaxtyped(typechecker=beartype)
    def __call__(self, x: Float[Array, "b n h"]) -> Float[Array, "b n h"]:
        q = self.Wq(x)
        k = self.Wk(x)
        v = self.Wv(x)

        # Split into heads: (b, n, h) -> (b, heads, n, head_size)
        q = rearrange(q, 'b n (heads d) -> b heads n d', heads=self.n_heads)
        k = rearrange(k, 'b n (heads d) -> b heads n d', heads=self.n_heads)
        v = rearrange(v, 'b n (heads d) -> b heads n d', heads=self.n_heads)

        scale = self.head_size ** -0.5

        # For QKᵀ we want (n, head_size) @ (head_size, n)
        k = rearrange(k, 'b heads n d -> b heads d n')
        attn_weights = jnp.matmul(q, k) * scale

        # Causal mask:
        #     k0  k1  k2  k3
        # q0 [■   □   □   □ ]
        # q1 [■   ■   □   □ ]
        # q2 [■   ■   ■   □ ]
        # q3 [■   ■   ■   ■ ]
        seq_len = x.shape[1]
        causal_mask = jnp.tril(jnp.ones((seq_len, seq_len), dtype=jnp.bool_))

        # □ become -∞
        attn_weights = jnp.where(causal_mask, attn_weights, float('-inf'))

        attn_weights = jax.nn.softmax(attn_weights, axis=-1)

        out = jnp.matmul(attn_weights, v)

        # Merge heads: (b, heads, n, head_size) -> (b, n, h)
        out = rearrange(out, 'b heads n d -> b n (heads d)')

        return self.Wo(out)

### The MLP Block

The MLP block, otherwise known as a feed forward network. As previously mentioned, the MLP holds two-thirds of the LLM's weights and is also where "information" is stored.

Let's say the words "Micheal Jordan" is inputted, attention would create the relationship that Michael before Jordan is a significant relationship, but the MLP learns who he is and what achievements he has.

<div align="center">
    <img src="https://raw.githubusercontent.com/chloeli-15/ARENA_img/main/img/transformer-mlp-new-2.png" alt="Image" width="550" height="400" />
</div>

In [ ]:
class MultiLayerPerceptron(nnx.Module):
    def __init__(self, config: GPT2Config, rngs: nnx.Rngs):
        self.fc1 = nnx.Linear(config.hidden_size, config.mlp_hidden_size, rngs=rngs)
        self.fc2 = nnx.Linear(config.mlp_hidden_size, config.hidden_size, rngs=rngs)

    @jaxtyped(typechecker=beartype)
    def __call__(self, x: Float[Array, "b n h"]) -> Float[Array, "b n h"]:
        x = self.fc1(x)
        x = jax.nn.gelu(x)
        x = self.fc2(x)
        return x

### Transformer Block

Finally, we get to the transformer block, which we'll create a list of, for our GPT-2 model. In general, this same transformer block that we'll create can be used to make up different sizes of GPT-2.

<div align="center">
    <img src="https://raw.githubusercontent.com/chloeli-15/ARENA_img/main/img/transformer-block2.png" alt="Image" width="600" height="400" />
</div>


In [ ]:
class TransformerLayer(nnx.Module):
    def __init__(self, config: GPT2Config, rngs: nnx.Rngs):
        self.attn = MultiHeadSelfAttention(config, rngs)
        self.mlp = MultiLayerPerceptron(config, rngs)
        self.ln1 = nnx.LayerNorm(config.hidden_size, rngs=rngs)
        self.ln2 = nnx.LayerNorm(config.hidden_size, rngs=rngs)

    @jaxtyped(typechecker=beartype)
    def __call__(self, x: Float[Array, "b n h"]) -> Float[Array, "b n h"]:
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

### Full Model!

In [ ]:
class GPT2(nnx.Module):
    def __init__(self, config: GPT2Config, rngs: nnx.Rngs):
        self.wte = nnx.Embed(config.vocab_size, config.hidden_size, rngs=rngs)
        self.wpe = nnx.Embed(config.max_seq_len, config.hidden_size, rngs=rngs)
        self.layers = [TransformerLayer(config, rngs) for _ in range(config.n_layers)]
        self.ln_f = nnx.LayerNorm(config.hidden_size, rngs=rngs)

    @jaxtyped(typechecker=beartype)
    def __call__(self, input_ids: Integer[Array, "b n"]) -> Float[Array, "b n v"]:
        batch_size, seq_len = input_ids.shape

        positions = jnp.arange(seq_len)

        # Token Embeddings + Positional Embeddings
        x = self.wte(input_ids) + self.wpe(positions)

        # Pass the hidden state through our layers
        for layer in self.layers:
            x = layer(x)

        x = self.ln_f(x)

        # Weight tying!
        logits = x @ self.wte.embedding.T

        return logits

### Generating a Sentence!

In [ ]:
@jaxtyped(typechecker=beartype)
def generate(model: GPT2, prompt: str, max_new_tokens: int = 20) -> str:
    input_ids: Integer[Array, "1 n"] = jnp.array(tokenizer(prompt, return_tensors="np")["input_ids"])

    for _ in range(max_new_tokens):
        logits: Float[Array, "1 n v"] = model(input_ids)
        next_token_logits: Float[Array, "1 v"] = logits[:, -1, :]
        next_token: Integer[Array, "1 1"] = jnp.argmax(next_token_logits, axis=-1, keepdims=True)
        input_ids: Integer[Array, "1 n"]  = jnp.concatenate([input_ids, next_token], axis=1)

    return tokenizer.decode(input_ids[0])

In [ ]:
# @title ##### Currently Random (Dumb) Model

config = GPT2Config()
dumb_model = GPT2(config, rngs=nnx.Rngs(0))
print(generate(dumb_model, "The quick brown fox"))

The quick brown foxinioninion chef chefulativeulative barriers barriers barriers barriers barriers barriers barriers barriers barriers barriers barriers barriers barriers barriers


## Training

In [ ]:
STEPS = 100_000
LOG_EVERY = 100
SAVE_EVERY = 50_000

model = GPT2(config, rngs=nnx.Rngs(0))

schedule = optax.warmup_cosine_decay_schedule(
    init_value=0.0,
    peak_value=6e-4,
    warmup_steps=2_000,
    decay_steps=STEPS,
    end_value=6e-5,
)

### CODE HERE: Define an Optimizer
tx = optax.adamw(learning_rate=schedule, weight_decay=0.1)
optimizer = nnx.Optimizer(model, tx, wrt=nnx.Param)

In [ ]:
# @title #### Pre-processing our Dataset
def make_batches(ds, batch_size: int = 32):
    def tokenize_and_chunk(batch):
        tokens = tokenizer(batch["text"])
        all_ids = np.concatenate([np.array(ids) for ids in tokens["input_ids"]])
        total = (len(all_ids) // (config.max_seq_len + 1)) * (config.max_seq_len + 1)
        chunks = all_ids[:total].reshape(-1, config.max_seq_len + 1)
        return {"input_ids": chunks}

    ds = ds.filter(lambda x: len(x["text"]) > 0)
    ds = ds.map(
        tokenize_and_chunk,
        batched=True,
        batch_size=1000,
        remove_columns=["text"],
    )
    ds.set_format("numpy")
    return ds.iter(batch_size=batch_size)

In [ ]:
import wandb
wandb.init(project="gpt2-jax")

def log_metrics(step, loss):
    wandb.log({"loss": float(loss), "step": int(step)})


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 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


wandb: WARNING Invalid choice
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: gabehubner (gabehubner-florida-international-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [ ]:
@nnx.jit(static_argnames=("config",))
def train_step(
    model: GPT2,
    optimizer: nnx.Optimizer,
    input_ids: Integer[Array, "b n"],
    config: GPT2Config,
    step: Integer[Array, ""]
) -> Float[Array, ""]:

    def loss_fn(model: GPT2) -> Float[Array, ""]:
        # ■: Labels.
        #         the_  and_ ...  boy_
        # l1  [    ■    □    ...   □  ]
        # l2  [    □    □    ...   ■  ]
        # l3  [    □    ■    ...   □  ]
        # l4  [    ■    □    ...   □  ]
        logits = model(input_ids[:, :-1])
        targets = input_ids[:, 1:]

        # logits -> (batch_size * (seq_len - 1), vocab_size)
        # targets -> (batch_size * (seq_len - 1))
        loss = optax.softmax_cross_entropy_with_integer_labels(
            logits.reshape(-1, config.vocab_size),
            targets.reshape(-1),
        ).mean()
        return loss

    loss, grads = nnx.value_and_grad(loss_fn)(model)
    optimizer.update(model, grads)

    jax.experimental.io_callback(log_metrics, (), step, loss, ordered=False)
    return loss

In [ ]:
import os

checkpoint_path = "/content/drive/MyDrive/gpt2_ckpt"
options = ocp.CheckpointManagerOptions(max_to_keep=2, create=True)
mngr = ocp.CheckpointManager(
    checkpoint_path,
    options=options,
    item_names=('state',)
)

def save_checkpoint(step: int):
    _, state = nnx.split(model)

    mngr.save(
        step,
        args=ocp.args.Composite(
            state=ocp.args.StandardSave(state)
        )
    )
    print(f"Checkpoint saved for step {step}")

def load_checkpoint(step: int = None):
    target_step = step if step is not None else mngr.latest_step()

    if target_step is None:
        print("No checkpoints found.")
        return None

    graphdef, state = nnx.split(model)

    restored = mngr.restore(
        target_step,
        args=ocp.args.Composite(
            state=ocp.args.StandardRestore(state)
        )
    )

    print(f"Checkpoint restored from step {target_step}")
    return nnx.merge(graphdef, restored['state'])

In [ ]:
batches = make_batches(ds.select(range(len(ds))), batch_size=32)
pbar = tqdm(zip(range(STEPS//10000), batches), total=STEPS//10000, desc="training")

for step, batch in pbar:
    input_ids = jnp.array(batch["input_ids"])
    loss = train_step(model, optimizer, input_ids, config, jnp.int32(step))

    pbar.set_postfix(loss=f"{loss:.4f}", step=step)

    if step % SAVE_EVERY == 0 and step > 0:
        save_checkpoint(step)

mngr.wait_until_finished()

### Let's Test Our Model!

In [ ]:
model = load_checkpoint()
print(generate(model, "The meaning of life is", max_new_tokens=100))
total = sum(x.size for x in jax.tree.leaves(nnx.state(model)))
print(f"Total parameters: {total:,}")

Checkpoint restored from step 50000
The meaning of life is that we are all in the same place. We are all in the same place. We are all in the same place. We are all in the same place. We are all in the same place. We are all in the same place. We are all in the same place. We are all in the same place. We are all in the same place. We are all in the same place. We are all in the same place. We are all in the same place. We are all
Total parameters: 124,402,944


And we're done! 🥳🎉